<a href="https://colab.research.google.com/github/AAwaisYaseen/computer-vision-logistics/blob/main/COMP6011_pidnet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# For PIDNET S
# https://drive.google.com/file/d/1wnMVQOmTEkSm8dB2A9TsSHG-yCgpGN0b/view?usp=drive_link

# For PIDNET M
# https://drive.google.com/file/d/1nCBp6OC8uXYdzDsSmhdw5a5zUF5Mce-6/view?usp=drive_link

# https://github.com/xujiacong/pidnet?tab=readme-ov-file

# weight link for PIDNET cityscapes
# https://drive.google.com/drive/folders/0BySIOtxxULinfld0LTcxYndTbFpWNjVpWm9nREU1T3hJUW5IS2otOUJDMmtnZERuODFPVU0?resourcekey=0-nauDQNE1efkunvcg89ZlDA

In [ ]:
!pip install torch torchvision --quiet
!pip install numpy pillow tqdm matplotlib --quiet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile
import os

ZIP_PATH = '/content/drive/MyDrive/AAIRT1/sydneyscapes.zip'
EXTRACT_PATH = '/content/sydneyscapes'

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)
print('Done!')

DATASET_PATH = '/content/sydneyscapes/sydneyscapes'
print('DATASET_PATH:', DATASET_PATH)

Mounted at /content/drive
Done!
DATASET_PATH: /content/sydneyscapes/sydneyscapes


In [ ]:
import os
import time
import numpy as np
from PIL import Image
import torch
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


In [ ]:
with open(os.path.join(DATASET_PATH, 'val.txt'), 'r') as f:
    val_files = [line.strip() for line in f.readlines()]

print(f'Total validation images: {len(val_files)}')

Total validation images: 152


In [ ]:
def compute_miou_direct(preds, labels, num_classes=19, ignore_index=255):
    iou_list = []
    for cls in range(num_classes):
        pred_mask = preds == cls
        label_mask = labels == cls
        valid_mask = labels != ignore_index
        intersection = (pred_mask & label_mask & valid_mask).sum()
        union = ((pred_mask | label_mask) & valid_mask).sum()
        if union == 0:
            continue
        iou_list.append(intersection / union)
    return np.mean(iou_list) if iou_list else 0.0

print('mIoU function ready.')

mIoU function ready.


In [ ]:
transform = transforms.Compose([
    transforms.Resize((512, 1024)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
print('Transform ready.')

Transform ready.


In [ ]:
!git clone https://github.com/XuJiacong/PIDNet.git
%cd /content/PIDNet
print('Done:', os.listdir('/content/PIDNet'))

Cloning into 'PIDNet'...
remote: Enumerating objects: 398, done.
remote: Counting objects: 100% (201/201), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 398 (delta 139), reused 128 (delta 128), pack-reused 197 (from 2)
Receiving objects: 100% (398/398), 212.80 MiB | 24.17 MiB/s, done.
Resolving deltas: 100% (192/192), done.
/content/PIDNet
Done: ['figs', 'datasets', 'tools', 'utils', 'data', 'configs', '.git', 'pretrained_models', 'LICENSE', 'models', 'samples', 'README.md']


In [ ]:
import gdown
import os

os.makedirs('/content/PIDNet/pretrained_models/cityscapes', exist_ok=True)

print('Downloading PIDNet-S...')
gdown.download(
    'https://drive.google.com/file/d/1wnMVQOmTEkSm8dB2A9TsSHG-yCgpGN0b/view?usp=drive_link',
    '/content/PIDNet/pretrained_models/cityscapes/PIDNet_S_Cityscapes_val.pt',
    fuzzy=True
)

print('Downloading PIDNet-M...')
gdown.download(
    'https://drive.google.com/file/d/1nCBp6OC8uXYdzDsSmhdw5a5zUF5Mce-6/view?usp=drive_link',
    '/content/PIDNet/pretrained_models/cityscapes/PIDNet_M_Cityscapes_val.pt',
    fuzzy=True
)

print('PIDNet-S exists:', os.path.exists('/content/PIDNet/pretrained_models/cityscapes/PIDNet_S_Cityscapes_val.pt'))
print('PIDNet-M exists:', os.path.exists('/content/PIDNet/pretrained_models/cityscapes/PIDNet_M_Cityscapes_val.pt'))

Downloading...
From (original): https://drive.google.com/uc?id=1wnMVQOmTEkSm8dB2A9TsSHG-yCgpGN0b
From (redirected): https://drive.google.com/uc?id=1wnMVQOmTEkSm8dB2A9TsSHG-yCgpGN0b&confirm=t&uuid=90b17fc9-e4e7-41ba-a29a-da8030b28812
To: /content/PIDNet/pretrained_models/cityscapes/PIDNet_S_Cityscapes_val.pt
100%|██████████| 31.1M/31.1M [00:00<00:00, 70.3MB/s]


Downloading...
From (original): https://drive.google.com/uc?id=1nCBp6OC8uXYdzDsSmhdw5a5zUF5Mce-6
From (redirected): https://drive.google.com/uc?id=1nCBp6OC8uXYdzDsSmhdw5a5zUF5Mce-6&confirm=t&uuid=5f947b3f-10fb-4d33-a8f1-d4eee60c1834
To: /content/PIDNet/pretrained_models/cityscapes/PIDNet_M_Cityscapes_val.pt
100%|██████████| 139M/139M [00:02<00:00, 66.3MB/s]

PIDNet-S exists: True
PIDNet-M exists: True
